<a href="https://colab.research.google.com/github/jdnogues/DIO/blob/main/Desafio_DIO_Criando_um_sistema_de_assist%C3%AAncia_virtual_do_zero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Projeto: Assistente Virtual com PLN no Google Colab**

**Objetivo:** Desenvolver um assistente virtual capaz de ouvir comandos de voz, transcrevê-los para texto, executar ações pré-definidas e responder com áudio.

**Passo 1: Instalação das Bibliotecas Necessárias**

Primeiro, vamos instalar todas as ferramentas que precisaremos.

In [1]:
# --- Célula de Instalação ---

# Para Text-to-Speech (TTS)
!pip install -q gTTS

# Para Speech-to-Text (STT)
!pip install -q SpeechRecognition

# Para as ações do assistente
!pip install -q Wikipedia-API

print("\nBibliotecas instaladas com sucesso! Por favor, reinicie o ambiente de execução.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 32.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

Bibliotecas instaladas com sucesso! Por favor, reinicie o ambiente de execução.


# **Ação Obrigatória:** Após esta célula executar, **reinicie o ambiente de execução** (Ambiente de execução > Reiniciar sessão).

**Módulo 1: Text-to-Speech (TTS)**

Esta parte será responsável por dar "voz" ao nosso assistente. Usaremos a biblioteca gTTS (Google Text-to-Speech), que é simples e eficaz.

**Passo 2:** Função para Falar

In [13]:
from gtts import gTTS
from IPython.display import Audio, display
import time # Importa a biblioteca de tempo

def falar(texto):
    """
    Converte um texto em áudio, o reproduz e ESPERA a fala terminar.
    """
    try:
        # Cria o objeto gTTS com o texto e o idioma corrigido para 'pt'
        tts = gTTS(text=texto, lang='pt')
        tts.save("assistente_fala.mp3")

        # Calcula uma duração estimada da fala (ex: 0.1 segundo por caractere + 0.5s de buffer)
        duracao_estimada = (len(texto) * 0.08) + 0.5

        print(f"Assistente: {texto}")
        display(Audio("assistente_fala.mp3", autoplay=True))

        # Pausa a execução do script pela duração estimada do áudio
        time.sleep(duracao_estimada)

    except Exception as e:
        print(f"Ocorreu um erro no módulo de fala: {e}")

# --- Teste do Módulo de Fala Sincronizado ---
falar("Esta é a primeira frase.")
falar("Esta é a segunda frase, que agora só começa após a primeira terminar.")

Assistente: Esta é a primeira frase.


Assistente: Esta é a segunda frase, que agora só começa após a primeira terminar.


**Módulo 2: Speech-to-Text (STT) e Execução de Comandos**

Esta é a parte central. O assistente irá ouvir, transcrever o áudio para texto e, em seguida, interpretar o comando.

**Passo 3:** Função para Ouvir e Transcrever
Para capturar áudio do microfone no Colab, precisamos de um pequeno trecho de código JavaScript, pois o Python no servidor do Colab não tem acesso direto ao seu microfone.

In [14]:
def ouvir_comando(tempo_de_gravacao=5): # O valor padrão é 5 segundos
    """
    Captura, converte e transcreve o áudio do microfone.
    """
    try:
        print(f"Ouvindo por {tempo_de_gravacao} segundos...")
        display(Javascript(RECORD))
        # Usa o parâmetro para definir o tempo de gravação
        s = output.eval_js(f'record({tempo_de_gravacao * 1000})')
        b = b64decode(s.split(',')[1])

        audio_original = "comando_audio_original.webm"
        with open(audio_original, "wb") as f:
            f.write(b)

        audio_convertido_wav = "comando_audio.wav"
        subprocess.run(['ffmpeg', '-i', audio_original, '-ac', '1', '-ar', '16000', audio_convertido_wav, '-y', '-hide_banner', '-loglevel', 'error'])

        r = sr.Recognizer()
        with sr.AudioFile(audio_convertido_wav) as source:
            audio_data = r.record(source)

        texto_comando = r.recognize_google(audio_data, language='pt-BR')
        print(f"Você disse: {texto_comando}")
        return texto_comando.lower()

    except sr.UnknownValueError:
        return ""
    except Exception as e:
        print(f"Ocorreu um erro na captura ou transcrição: {e}")
        return ""

**Passo 4: Processando os Comandos**

Agora, criamos a lógica que verifica o texto transcrito e decide qual ação tomar.

In [15]:
import wikipediaapi
import requests
from IPython.display import HTML, display

def processar_comandos(comando):
    """
    Interpreta o comando de texto e executa a ação correspondente.
    """
    if not comando:
        falar("Não consegui entender. Poderia repetir?")
        return

    # Comando para Wikipedia
    if "pesquisar por" in comando and "wikipédia" in comando:
        termo = comando.split("pesquisar por")[-1].replace("na wikipédia", "").strip()
        falar(f"Pesquisando por {termo} na Wikipédia.")

        wiki_pt = wikipediaapi.Wikipedia('pt')
        pagina = wiki_pt.page(termo)

        if pagina.exists():
            falar("Aqui está um resumo do que encontrei.")
            print(f"\n--- Resumo da Wikipedia: {pagina.title} ---")
            print(pagina.summary[0:500] + "...")
        else:
            falar(f"Desculpe, não encontrei uma página sobre {termo}.")

    # Comando para YouTube
    elif "tocar" in comando and "youtube" in comando:
        video = comando.split("tocar")[-1].replace("no youtube", "").strip()
        falar(f"Encontrei este vídeo sobre {video} no YouTube.")

        url_busca = f"https://www.youtube.com/embed?listType=search&list={video.replace(' ', '+')}"
        display(HTML(f'<iframe width="560" height="315" src="{url_busca}" frameborder="0" allowfullscreen></iframe>'))

    # Comando para Previsão do Tempo
    elif "previsão do tempo para" in comando:
        cidade = comando.split("previsão do tempo para")[-1].strip()
        falar(f"Verificando a previsão do tempo para {cidade}.")

        try:
            # Adicionamos '&lang=pt' para pedir a tradução
            url = f"https://wttr.in/{cidade}?format=j1&lang=pt"
            response = requests.get(url)
            data = response.json()

            condicao_atual = data['current_condition'][0]
            temperatura = condicao_atual['temp_C']
            descricao = condicao_atual['weatherDesc'][0]['value']

            resposta = f"A previsão do tempo para {cidade} é de {descricao}, com temperatura de {temperatura} graus Celsius."
            falar(resposta)

        except Exception as e:
            falar(f"Desculpe, não consegui encontrar a previsão do tempo para {cidade}.")
            print(f"Erro na API de tempo: {e}")

    else:
        falar("Comando não reconhecido. Tente 'pesquisar por...', 'tocar...' ou 'previsão do tempo para...'.")

**Passo 5: Juntando Tudo - O Loop Principal**

Finalmente, criamos um loop que mantém o assistente ativo, ouvindo por uma "palavra de ativação" (wake word).

In [16]:
# Inicia a assistente
falar("Assistente virtual iniciada. Diga 'assistente' para me ativar.")

# Loop para aguardar a palavra de ativação
while True:
    print("\nAguardando palavra de ativação...")
    # Grava por 3 segundos para a palavra de ativação
    comando_ativacao = ouvir_comando(tempo_de_gravacao=3)

    if "assistente" in comando_ativacao: # Palavra de ativação
        falar("Sim? O que você deseja?")

        # Grava por 7 segundos para o comando completo
        comando_completo = ouvir_comando(tempo_de_gravacao=7)

        # Processa o comando se algo foi dito
        if comando_completo:
            processar_comandos(comando_completo)
            falar("Posso ajudar com mais alguma coisa?")
        else:
            falar("Não ouvi o comando. Se precisar, me chame de novo.")

    # Para interromper o loop, pare a execução da célula manualmente no Colab.

Assistente: Assistente virtual iniciada. Diga 'assistente' para me ativar.



Aguardando palavra de ativação...
Ouvindo por 3 segundos...


<IPython.core.display.Javascript object>

Você disse: estante

Aguardando palavra de ativação...
Ouvindo por 3 segundos...


<IPython.core.display.Javascript object>

...

Assistente: A previsão do tempo para vitória é de Cloudy, com temperatura de 21 graus Celsius.


Assistente: Posso ajudar com mais alguma coisa?



Aguardando palavra de ativação...
Ouvindo por 3 segundos...


<IPython.core.display.Javascript object>

KeyboardInterrupt: 